> 对应 Lec 9-11（优化算法部分）。覆盖梯度下降/牛顿法/L-BFGS/SGD/Adam 的通用实现模板，以及分布式 Mini-batch SGD 和 Ring All-Reduce 通信模式。具体到 Logistic 回归的分布式实现见"5. 分类问题_实现.md"；算法概念见"6. 光滑优化_概念解释.md"。

---

## 梯度下降（GD）通用模板

### 算法思路

梯度下降的核心更新公式只有一行：

$$\beta^{(t+1)} = \beta^{(t)} - \alpha \nabla\ell(\beta^{(t)})$$

每一步沿负梯度方向走一小步。判断收敛通常用梯度范数 $\|\nabla\ell(\beta)\|$ 是否小于阈值——梯度接近 0 意味着已经接近极值点（驻点）。这是所有后面方法（牛顿法、L-BFGS、SGD、Adam）的最基础版本，写代码时建议先把这个模板写熟，因为其余方法都是在"算什么方向、走多大步"上做改造。

### 基础 GD 框架（固定学习率，梯度范数收敛判断）

In [ ]:
import numpy as np

def gradient_descent(grad_func, beta_init, lr=0.01, tol=1e-6, max_iters=1000):
    """
    通用梯度下降模板。

    Args:
        grad_func: 函数 grad_func(beta) -> 梯度向量（调用者自己实现，
                   通常封装了具体模型的梯度公式，如 Logistic 回归的 X^T(p-y)）。
        beta_init: 初始参数向量。
        lr: 学习率（步长），需要手动调参。
        tol: 梯度范数的收敛阈值。
        max_iters: 最大迭代次数（防止不收敛时死循环）。

    Returns:
        beta: 收敛后的参数估计。
    """
    beta = beta_init.copy()
    for t in range(max_iters):
        grad = grad_func(beta)
        # 梯度范数足够小 -> 已接近驻点，提前退出
        if np.linalg.norm(grad) < tol:
            print(f"GD 收敛于第 {t} 次迭代")
            break
        beta = beta - lr * grad
    return beta

### 带学习率衰减的 GD（步长调度策略）

**思路解释**：固定学习率有个两难——太大会在最优点附近震荡不收敛，太小则前期收敛极慢。常见折中是让学习率随迭代次数衰减，前期大步快速逼近，后期小步精细收敛。

In [ ]:
def gradient_descent_decay(grad_func, beta_init, lr0=0.1, decay=0.01,
                            tol=1e-6, max_iters=1000):
    """
    学习率衰减版 GD：alpha_t = lr0 / (1 + decay * t)。
    decay 越大，学习率衰减越快。
    """
    beta = beta_init.copy()
    for t in range(max_iters):
        grad = grad_func(beta)
        if np.linalg.norm(grad) < tol:
            break
        lr_t = lr0 / (1.0 + decay * t)   # 随 t 增大，步长单调减小
        beta = beta - lr_t * grad
    return beta

### 线搜索（Armijo 条件）简化实现

**思路解释**：与固定/预设衰减学习率不同，线搜索是在每一步**当场**寻找一个"足够好"的步长，而不是提前订好规则。Armijo 条件要求新点的函数值比当前点至少下降了"预期下降量的一部分"：

$$\ell(\beta - \alpha\nabla\ell(\beta)) \leq \ell(\beta) - c_1 \alpha \|\nabla\ell(\beta)\|^2$$

实现上常用 **回溯线搜索（backtracking）**：从一个较大的步长开始，不满足条件就乘上一个收缩因子（如 0.5）反复缩小，直到满足为止。

In [ ]:
def backtracking_line_search(loss_func, grad_func, beta, grad,
                              alpha0=1.0, shrink=0.5, c1=1e-4, max_trials=50):
    """
    Armijo 回溯线搜索：寻找满足充分下降条件的步长 alpha。

    loss_func(beta) -> 标量损失值（用于判断下降是否足够）。
    grad: 当前点的梯度（已经算好，避免重复计算）。
    """
    alpha = alpha0
    loss_current = loss_func(beta)
    grad_sq_norm = grad @ grad   # ||grad||^2，提前算好避免重复计算
    for _ in range(max_trials):
        beta_new = beta - alpha * grad
        # Armijo 条件：新点的损失下降是否达到预期的 c1 倍
        if loss_func(beta_new) <= loss_current - c1 * alpha * grad_sq_norm:
            return alpha
        alpha *= shrink   # 不满足就缩小步长，继续尝试
    return alpha  # 达到最大尝试次数，返回当前（很小的）步长

---

## 牛顿法通用模板

### 算法思路

牛顿法用二阶信息（Hessian）代替一阶信息，更新公式：

$$\beta^{(t+1)} = \beta^{(t)} - \alpha [H^{(t)}]^{-1}\nabla\ell(\beta^{(t)})$$

直觉：梯度下降只知道"哪个方向下降最快"，牛顿法还知道"这个方向上曲面有多弯"，因此能一步走到更精确的位置。代价是每步要算 Hessian（$p\times p$ 矩阵）并解一个线性方程组。

### 牛顿方向：np.linalg.solve(H, grad) 而非 H^{-1} @ grad

**为什么不能直接求逆？** 这是"四条矩阵计算原则"里讲过的同一条规则：显式求逆 $H^{-1}$ 的代价是 $O(p^3)$ 且数值不稳定（条件数差时误差被放大），而 `solve` 直接求解线性方程组 $Hx=g$，同样是 $O(p^3)$ 但数值更稳定，且不需要先构造出完整的逆矩阵。

In [ ]:
import numpy as np

# ❌ 错误（显式求逆，数值不稳定且浪费）
delta = np.linalg.inv(H) @ grad

# ✅ 正确（解线性方程组）
delta = np.linalg.solve(H, grad)

### 带步长的牛顿更新（α=1 的条件与退化为 GD 的情形）

In [ ]:
def newton_method(grad_func, hess_func, beta_init, tol=1e-6, max_iters=50, alpha=1.0):
    """
    通用牛顿法模板。

    Args:
        grad_func: beta -> 梯度向量
        hess_func: beta -> Hessian 矩阵（p×p）
        alpha: 步长。理论上离最优点足够近时 alpha=1 是"天然步长"，
               不需要调参；离最优点较远时 alpha=1 可能发散，
               这时退化成阻尼牛顿法（damped Newton），用 alpha<1 或线搜索。
    """
    beta = beta_init.copy()
    for t in range(max_iters):
        grad = grad_func(beta)
        if np.linalg.norm(grad) < tol:
            print(f"牛顿法收敛于第 {t} 次迭代")
            break
        H = hess_func(beta)
        delta = np.linalg.solve(H, grad)   # 牛顿方向
        beta = beta - alpha * delta
    return beta

**何时退化为 GD？** 如果用单位矩阵 $I$ 代替 Hessian（即假设曲面各方向曲率都一样），牛顿更新公式就变成 $\beta - \alpha I^{-1}\nabla\ell = \beta - \alpha\nabla\ell$，正是梯度下降。这说明 GD 可以看成牛顿法的一种"曲率信息缺失"的特例。

### 二阶收敛验证（梯度范数每步平方衰减）

**思路解释**：牛顿法的标志性性质是**二阶收敛**——当迭代点足够接近最优点时，误差按照"平方"速度缩小，即 $\|\beta^{(t+1)} - \beta^*\| \approx C\|\beta^{(t)} - \beta^*\|^2$。直观验证方法：打印每步的梯度范数，应观察到类似 `0.1 -> 0.01 -> 0.0001 -> 0.00000001` 这种迭代次数极少、但下降极快的模式（与 GD 的线性收敛——每步按固定比例缩小——形成鲜明对比）。

In [ ]:
def newton_method_verbose(grad_func, hess_func, beta_init, max_iters=20):
    """带打印的牛顿法，用于在作业/考试中验证二阶收敛现象。"""
    beta = beta_init.copy()
    grad_norms = []
    for t in range(max_iters):
        grad = grad_func(beta)
        norm_g = np.linalg.norm(grad)
        grad_norms.append(norm_g)
        print(f"iter {t}: ||grad|| = {norm_g:.2e}")
        if norm_g < 1e-10:
            break
        H = hess_func(beta)
        beta = beta - np.linalg.solve(H, grad)
    return beta, grad_norms

---

## L-BFGS 调用模板（scipy 接口）

### 算法思路

L-BFGS（Limited-memory BFGS）不显式构造和存储 $p\times p$ 的 Hessian（或其逆），而是只保留最近 $m$ 步的"参数位移" $s^{(t)} = \beta^{(t+1)}-\beta^{(t)}$ 和"梯度变化" $y^{(t)} = \nabla\ell(\beta^{(t+1)})-\nabla\ell(\beta^{(t)})$，通过一个叫"双循环递归"的算法近似计算 $H^{-1}g$。空间和时间复杂度都是 $O(mp)$（$m$ 通常取 5~20），远小于牛顿法的 $O(p^2)$/$O(p^3)$。考试和作业里基本不需要手写双循环递归，**直接调用 scipy 接口即可**——理解原理、会调接口是重点。

### scipy.optimize.minimize(method='L-BFGS-B') 调用方式

In [ ]:
from scipy.optimize import minimize

def loss_and_grad(beta, X, y):
    """
    同时返回损失值和梯度。jac=True 时 scipy 要求该函数返回 (loss, grad) 元组，
    这样可以避免损失和梯度分别用两次前向计算（节省一半算力）。
    """
    z = X @ beta
    p = 1.0 / (1.0 + np.exp(-z))           # 简化写法，生产环境用 stable_sigmoid
    loss = np.sum(-y * z + np.log(1 + np.exp(z)))
    grad = X.T @ (p - y)
    return loss, grad

beta_init = np.zeros(X.shape[1])
result = minimize(
    fun=loss_and_grad,
    x0=beta_init,
    args=(X, y),
    method='L-BFGS-B',
    jac=True,            # 告诉 scipy：fun 返回的是 (loss, grad)，不需要再数值微分
    options={'maxcor': 10, 'maxiter': 100}
)
beta_hat = result.x
print(f"收敛: {result.success}, 迭代次数: {result.nit}")

### fun / jac 参数：损失函数与梯度函数的封装

**两种写法的区别**（容易在考试代码题中混淆）：

In [ ]:
# 写法一：fun 只返回损失，jac 单独传一个函数
result = minimize(fun=loss_func, x0=beta_init, jac=grad_func, method='L-BFGS-B')

# 写法二（推荐，效率更高）：fun 同时返回 (损失, 梯度)，jac=True
result = minimize(fun=loss_and_grad, x0=beta_init, jac=True, method='L-BFGS-B')

写法二更常用，因为损失和梯度的计算通常共享中间结果（如 $z = X\beta$、$p=\sigma(z)$），合并成一次调用可以避免重复计算。

### 历史对数 m（maxcor 参数）的设置与内存占用（O(mp)）

`maxcor` 对应理论里的 $m$——保留多少步历史位移/梯度变化。

- $m$ 越大：近似 Hessian 更精确，收敛步数可能更少，但每步计算和内存开销变大（$O(mp)$）。
- $m$ 越小（如 3~5）：内存占用小，适合 $p$ 极大的场景，但近似精度下降。
- 默认值（scipy 中 `maxcor=10`）在大多数问题上是合理的折中，一般不需要调。

---

## Mini-batch SGD 实现模板

### 算法思路

全量梯度下降每步要扫描全部 $n$ 个样本才能更新一次参数，当 $n$ 极大时效率低。Mini-batch SGD 每步只用一个小批量（$B \ll n$）样本估计梯度，用"梯度噪声大但更新频率高"换取整体训练速度。$B$ 越大越接近全量 GD（梯度准、更新慢），$B$ 越小越接近纯 SGD（梯度噪、更新快）。

### mini-batch 抽取（np.random.choice 无放回，每 epoch shuffle）

In [ ]:
def mini_batch_sgd(grad_func, beta_init, X, y, batch_size=64,
                    n_epochs=10, lr0=0.1, decay=0.01):
    n = X.shape[0]
    beta = beta_init.copy()
    t = 0  # 全局步数，用于学习率衰减
    for epoch in range(n_epochs):
        # 每个 epoch 开始前打乱样本顺序（shuffle），避免按固定顺序训练带来的偏差
        perm = np.random.permutation(n)
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]   # 无放回地切出一个 mini-batch
            X_batch, y_batch = X[idx], y[idx]

            grad = grad_func(beta, X_batch, y_batch)
            lr_t = lr0 / (1.0 + decay * t)          # Robbins-Monro 型学习率衰减
            beta = beta - lr_t * grad
            t += 1
    return beta

### 单步 SGD 更新框架（计算局部梯度 → 更新 β）

In [ ]:
def sgd_step(beta, X_batch, y_batch, lr, grad_func):
    """单步更新：计算 mini-batch 梯度 -> 沿负梯度方向走一步。"""
    grad = grad_func(beta, X_batch, y_batch)
    return beta - lr * grad

### Robbins-Monro 学习率衰减：α_t = α_0 / (1 + decay * t)

**为什么需要这种特定形式的衰减？** Robbins-Monro 条件要求学习率序列同时满足：

$$\sum_t \alpha_t = \infty \quad（步长之和发散，保证能走到任意远的地方）$$
$$\sum_t \alpha_t^2 < \infty \quad（步长平方和收敛，保证最终能稳定下来不震荡）$$

$\alpha_t = \alpha_0/(1+\text{decay}\cdot t)$ 是 $\alpha_t = C/t$ 这一调和级数型衰减的变体，同时满足两个条件（调和级数 $\sum 1/t$ 发散，但 $\sum 1/t^2$ 收敛）。如果用**固定**学习率，第二个条件不满足，SGD 会在最优点附近持续震荡，无法精确收敛——但工程实践中常态化地接受这种近似（深度学习里几乎都用固定或分段衰减的学习率）。

---

## Momentum SGD 实现模板

### 算法思路

纯 SGD 的梯度噪声很大，更新方向经常"抖动"。Momentum 的思路是引入一个"速度"变量，把历史梯度做指数加权平均，让更新方向更平滑——类似小球在带摩擦的斜面上滚动，会积累惯性，不会被瞬间的小坑洼带偏方向。

### 速度向量初始化与更新：v ← γv + α∇ℓ_B

In [ ]:
def momentum_sgd(grad_func, beta_init, X, y, batch_size=64,
                  n_epochs=10, lr=0.01, gamma=0.9):
    n = X.shape[0]
    beta = beta_init.copy()
    v = np.zeros_like(beta)   # 速度向量，初始为 0

    for epoch in range(n_epochs):
        perm = np.random.permutation(n)
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            grad = grad_func(beta, X[idx], y[idx])

            # 速度更新：历史速度的 gamma 倍 + 当前梯度的贡献
            v = gamma * v + lr * grad
            # 参数更新：沿速度方向走（而不是直接沿梯度方向）
            beta = beta - v
    return beta

### 参数更新：β ← β - v

注意更新用的是速度 `v`，不是梯度本身——这是 Momentum 与普通 SGD 的唯一区别。

### γ=0.9 的常用默认值

$\gamma$（动量系数）控制历史速度的"记忆长度"。$\gamma=0.9$ 意味着大约保留前 $1/(1-\gamma)=10$ 步的梯度信息做平均，是深度学习里最常见的默认值；$\gamma=0$ 时退化为纯 SGD。

---

## Adam 优化器实现模板

### 算法思路

Adam = Momentum（一阶矩，平滑方向）+ 自适应学习率（二阶矩，给每个参数维度配不同的有效步长）。类比：一个聪明的旅行者不仅记住历史前进方向（动量），还会根据每个方向的地形坡度自动调整步幅——陡峭方向走小步，平坦方向走大步。

### 一阶矩 m 更新：m ← β₁m + (1-β₁)g

### 二阶矩 v 更新：v ← β₂v + (1-β₂)g²（逐元素平方）

### 偏差修正：m̂ = m/(1-β₁^t)，v̂ = v/(1-β₂^t)

**为什么需要偏差修正？** $m, v$ 初始为 0，前几步更新后仍然严重偏向 0（比如第一步 $m^{(1)} = (1-\beta_1)g^{(1)}$，只有真实梯度的 $1-\beta_1=10\%$）。除以 $(1-\beta_1^t)$ 把这个系统性低估纠正回来；随着 $t$ 增大，$\beta_1^t \to 0$，修正项趋于 1，影响自然消失。

### 参数更新：β ← β - α·m̂/(√v̂ + ε)

### 默认超参数（β₁=0.9, β₂=0.999, ε=1e-8）

In [ ]:
class AdamOptimizer:
    """
    Adam 优化器（有状态对象，因为需要在多步迭代间持久保存 m, v, t）。
    """
    def __init__(self, beta_init, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.beta = np.array(beta_init, dtype=np.float64, copy=True)
        self.m = np.zeros_like(self.beta)   # 一阶矩
        self.v = np.zeros_like(self.beta)   # 二阶矩
        self.t = 0                          # 时间步，从 0 开始，第一次 step() 后变 1
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps

    def step(self, grad):
        """接收一次梯度，更新内部状态和参数。"""
        self.t += 1

        # 一阶矩、二阶矩的指数加权平均更新
        self.m = self.beta1 * self.m + (1 - self.beta1) * grad
        self.v = self.beta2 * self.v + (1 - self.beta2) * (grad ** 2)  # 逐元素平方，不是范数平方

        # 偏差修正
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)

        # 参数更新：注意 sqrt 在分母里只包住 v_hat，eps 在外面相加
        self.beta -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

    def get_params(self):
        return self.beta

**三个最容易写错的细节**（考试代码填空高频陷阱）：
1. `self.v += grad ** 2`（逐元素平方），不是 `grad @ grad`（标量内积）。
2. 更新公式分母是 `np.sqrt(v_hat) + eps`，不是 `np.sqrt(v_hat + eps)`——括号位置不同，含义完全不同。
3. `t` 从 0 开始，但**先 `t += 1` 再用 `beta1 ** t` 做偏差修正**，所以第一次调用时修正用的是 `beta1 ** 1`，不是 `beta1 ** 0`。

---

## 分布式 Mini-batch SGD（参数服务器模式）

### 算法思路

把"全量梯度的可加性"（$\nabla\ell(\beta)=\sum_i \nabla\ell_i(\beta)$）应用到 mini-batch 上：每轮先在头节点做一次**全局**随机抽样决定本轮要用哪些样本，再把抽样结果映射到各个工作节点的局部索引，让各节点只计算自己分到的那部分样本的梯度，最后头节点汇总求平均。这保证了"mini-batch 是全局均匀采样"而不是"只从某一个节点抽"，避免引入采样偏差。

### Driver 广播当前参数 β → Worker 并行计算小批量梯度

### Driver Reduce：全局梯度 = 均值(各 Worker 梯度)

### 每轮只传 β 和梯度（p 维向量），不传重传数据

In [ ]:
import ray

@ray.remote
def compute_minibatch_grad(X_chunk, y_chunk, beta, local_indices, grad_func):
    """在数据块上按给定的局部索引计算 mini-batch 梯度。"""
    if len(local_indices) == 0:
        return np.zeros_like(beta), 0   # 本轮没分到样本的节点返回零梯度
    X_batch = X_chunk[local_indices]
    y_batch = y_chunk[local_indices]
    grad = grad_func(beta, X_batch, y_batch)   # 返回的是求和后的梯度，不是均值
    return grad, len(local_indices)

def distributed_mini_batch_sgd(X_refs, y_refs, chunk_sizes, p, grad_func,
                                global_batch_size=1024, lr=0.01, n_steps=200):
    """
    参数服务器模式：头节点（driver）维护全局 beta，
    每轮广播 beta、收集各节点的局部梯度、汇总后更新。
    """
    beta = np.zeros(p)

    for step in range(n_steps):
        # 头节点做全局采样，映射到各数据块的局部索引（见 5. 分类问题_实现.md 的实现）
        local_indices = get_local_indices_from_global(global_batch_size, chunk_sizes)

        # Map：广播当前 beta，各节点并行计算局部梯度
        tasks = [
            compute_minibatch_grad.remote(X_refs[i], y_refs[i], beta, local_indices[i], grad_func)
            for i in range(len(X_refs)) if len(local_indices[i]) > 0   # 跳过本轮没分到样本的节点
        ]
        results = ray.get(tasks)   # Barrier：等待所有节点完成

        # Reduce：汇总梯度，除以总样本数得到平均梯度
        total_grad = sum(r[0] for r in results)
        total_n = sum(r[1] for r in results)
        beta = beta - lr * (total_grad / total_n)

    return beta

**关键设计**：每轮通信只传 `beta`（$p$ 维）和各节点返回的梯度（$p$ 维），数据本身已经提前用 `ray.put` 缓存到各节点，不会被重复传输。

---

## Ring All-Reduce 通信模式说明

### 算法思路

参数服务器模式有个结构性缺陷：所有工作节点都要把梯度发给同一个中心节点，中心节点的网络带宽随节点数 $K$ 线性增长，最终成为瓶颈。Ring All-Reduce 用"环形拓扑、无中心节点"解决这个问题——每个节点只与左右相邻节点通信，通过两阶段算法让所有节点最终都获得完整的全局梯度。

### Scatter-Reduce 阶段：每节点只累加 1/K 的梯度段

**思路解释**：把每个节点的梯度向量切成 $K$ 段（$K$ = 节点数）。第一阶段沿环形依次传递、累加，$K$ 轮通信后，每个节点恰好持有"某一段"梯度的**完整**求和结果（其余 $K-1$ 个节点对这一段的贡献已经在传递过程中累加进来）。

```
图示（4个节点，梯度切成4段 [a,b,c,d]）：
Node1: a1,b1,c1,d1   Node2: a2,b2,c2,d2   Node3: a3,b3,c3,d3   Node4: a4,b4,c4,d4

第1轮：每个节点把自己某一段发给下一个节点并累加
第2~K-1轮：继续沿环传递累加
最终：Node1 持有完整的 a 段之和，Node2 持有完整的 b 段之和，以此类推
```

### All-Gather 阶段：广播完整结果到所有节点

**思路解释**：第一阶段结束后，每个节点只有"一段"的完整结果，需要再做一轮沿环传递，把各自持有的完整段广播给所有节点，使每个节点最终都集齐全部 $K$ 段，即完整的全局梯度。

### 通信量 O(p) per node（与节点数 K 无关）的原因

每个节点在两阶段中，发送和接收的数据量各约为 $\frac{K-1}{K}\cdot p \approx p$（当 $K$ 较大时）。两阶段相加，每个节点总通信量约 $2p$——**只与梯度维度 $p$ 有关，与节点数 $K$ 无关**。这是 Ring All-Reduce 相对参数服务器的核心优势。

### 对比参数服务器：无单点通信瓶颈

| 架构 | 每节点通信量 | 中心瓶颈 |
|---|---|---|
| 单参数服务器 | PS 节点总接收量 $O(Kp)$ | 有，PS 节点带宽随 $K$ 线性增长 |
| Ring All-Reduce | $O(p)$，与 $K$ 无关 | 无，每节点只与相邻节点通信 |

Ring All-Reduce 是 PyTorch DDP、Horovod 等深度学习分布式训练框架的标准底层通信方案。